# V2D Challenge CPU Study - Dataset Audit

This notebook reads the downloaded V2D dataset from disk and reports the actual modalities, counts, coordinate conventions, camera metadata, and representative visualizations for all three tracks. Operations are bounded for an Apple Silicon CPU/MPS laptop with 16 GB RAM.

### Environment audit

**What it does.** Records the interpreter, package versions, MPS availability, and the environment fallback.

**Input.** Local Python runtime.

**Output.** `outputs/env_report.json` and a printed table.

**References.** PyTorch MPS documentation: https://pytorch.org/docs/stable/notes/mps.html

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

from importlib.metadata import version as pkg_version
packages=['numpy','pandas','pyarrow','matplotlib','opencv-python','trimesh','open3d','scipy','psutil','pin','nbformat','nbclient','torch']
versions={}
for p in packages:
    try: versions[p]=pkg_version(p)
    except Exception: versions[p]='unavailable'
try:
    import torch; mps=bool(torch.backends.mps.is_available()); tv=torch.__version__
except Exception as exc: mps=False; tv=f'error: {exc}'
probe=subprocess.run(['/opt/anaconda3/envs/v2d/bin/python','-c',"import importlib.util; print({m:bool(importlib.util.find_spec(m)) for m in ['numpy','pandas','pyarrow','matplotlib','cv2','trimesh','open3d','scipy','pinocchio','torch']})"],capture_output=True,text=True).stdout.strip()
env={'python_executable':sys.executable,'python_version':sys.version,'platform':platform.platform(),'cpu_only_scope':True,'mps_available':mps,'torch_version':tv,'package_versions':versions,'named_v2d_import_probe':probe,'v2d_environment_note':'The named v2d environment lacks pandas/pyarrow/nbformat in this machine image. Offline PyPI installation failed with DNS resolution; execution uses /opt/anaconda3/bin/python and records this fallback.'}
write_json(paths['root']/'env_report.json',env)
display(pd.DataFrame({'item':list(env),'value':[str(v)[:180] for v in env.values()]}))

### Track inventory and file structure

**What it does.** Enumerates actual Parquet, video, mesh, URDF, support-surface, and ground-plane files rather than assuming a remote layout.

**Input.** `video_to_data_challenge/track_1`, `track_2`, and `track_3`.

**Output.** `outputs/dataset_audit.json` and an inventory table.

**References.** Dataset card: https://huggingface.co/datasets/nvidia/video_to_data_challenge

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

t1=track1_paths(); t21=track2_paths(1); t22=track2_paths(2); t3=track3_paths()
def count(pattern,suffix=None):
 files=[p for p in (ROOT/pattern).glob('**/*') if p.is_file()]
 return sum(1 for p in files if suffix is None or p.suffix==suffix)
inventory={'track_1':{'episodes':len(t1),'parquets':count('video_to_data_challenge/track_1/data','.parquet'),'videos':count('video_to_data_challenge/track_1/videos','.mp4'),'fps':30.0,'frames_from_parquet':int(sum(len(read_parquet(r['parquet'])) for r in t1))},'track_2_tier_1':{'episodes':len(t21),'parquets':count('video_to_data_challenge/track_2/tier_1_multiview_caption/data','.parquet'),'videos':count('video_to_data_challenge/track_2/tier_1_multiview_caption/videos','.mp4'),'meshes':count('video_to_data_challenge/track_2/tier_1_multiview_caption/mesh','.glb'),'ground_planes':count('video_to_data_challenge/track_2/tier_1_multiview_caption/ground_plane','.json'),'support_surfaces':count('video_to_data_challenge/track_2/tier_1_multiview_caption/support_surface','.usda')},'track_2_tier_2':{'episodes':len(t22),'parquets':count('video_to_data_challenge/track_2/tier_2_synthetic_noise/data','.parquet'),'videos':count('video_to_data_challenge/track_2/tier_2_synthetic_noise/videos','.mp4'),'meshes':count('video_to_data_challenge/track_2/tier_2_synthetic_noise/mesh','.glb'),'ground_planes':count('video_to_data_challenge/track_2/tier_2_synthetic_noise/ground_plane','.json'),'support_surfaces':count('video_to_data_challenge/track_2/tier_2_synthetic_noise/support_surface','.usda')},'track_3':{'episodes':len(t3),'parquets':count('video_to_data_challenge/track_3/public/data','.parquet'),'videos':count('video_to_data_challenge/track_3/public/videos','.mp4'),'meshes':count('video_to_data_challenge/track_3/public/mesh','.glb'),'urdfs':count('video_to_data_challenge/track_3/public/urdf','.urdf'),'nominal_fps':20.25411910142744}}
write_json(paths['root']/'dataset_audit.json',inventory); display(pd.DataFrame(inventory).T.fillna(''))

### Track 1 representative video

**What it does.** Chooses the shortest third-person episode, samples frames, and records camera/object/task metadata.

**Input.** Track 1 metadata, Parquet index, and H.264 video.

**Output.** `outputs/track1_manifest.json` and `outputs/figs/track1_overview.png`.

**References.** Track 1 README: `video_to_data_challenge/track_1/README.md`

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

t1=track1_paths(); r=min(t1,key=lambda x:x['frame_count']); frames,vi=sample_video(r['video'],max_frames=12,resize_width=640)
manifest={**r,'video_info':vi,'gt_available':False,'gt_note':'Track 1 local release has frame indexes and videos only; no human/object pose or mesh ground truth.'}; write_json(paths['root']/'track1_manifest.json',manifest); save_contact_sheet(frames,paths['figs']/'track1_overview.png',[f"frame {i}" for i in vi['sample_indices']]); print('episode',r['episode_index'],'sequence',r.get('sequence_id'),'object',r.get('object'),'frames',r['frame_count'],'fps',f"{vi['fps']:.2f} Hz"); display(pd.DataFrame([{'episode':r['episode_index'],'camera':r.get('camera'),'object':r.get('object'),'duration_s':vi['duration_s'],'resolution_px':f"{vi['width']}x{vi['height']}",'gt':'held-out not distributed'}])); display(Image(filename=str(paths['figs']/'track1_overview.png')))

### Track 2 trajectories, assets, and coordinate convention

**What it does.** Reads both tiers for the same episode order and plots the released object trajectory against the fitted ground plane.

**Input.** Tier 1/tier 2 Parquets, ground-plane JSON, mesh/support-surface paths.

**Output.** `outputs/track2_manifest.json` and `outputs/figs/track2_overview.png`.

**References.** Track 2 README: `video_to_data_challenge/track_2/README.md`

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

t21=track2_paths(1); r=t21[0]; d=read_parquet(r['parquet']); p=np.vstack(d['observation.object.pose'].to_numpy()); plane=json.loads((ROOT/'video_to_data_challenge/track_2/tier_1_multiview_caption'/r['ground_plane']).read_text()); c=plane.get('plane',plane.get('coefficients',plane.get('ground_plane',[0,0,1,0]))); c=[c.get(k,0) for k in ('a','b','c','d')] if isinstance(c,dict) else c; dist=plane_signed_distance(p[:,:3],c); m={'episode':r,'rows':len(d),'pose_convention':'world_T_object, [x,y,z,qw,qx,qy,qz], metres and w-first quaternion','ground_plane_coefficients':c,'mesh_exists':(ROOT/'video_to_data_challenge/track_2/tier_1_multiview_caption'/r['mesh']).exists(),'support_surface_exists':(ROOT/'video_to_data_challenge/track_2/tier_1_multiview_caption'/r['support_surface']).exists()}; write_json(paths['root']/'track2_manifest.json',m); fig,ax=plt.subplots(2,1,figsize=(8,5),sharex=True); ax[0].plot(p[:,:3]); ax[0].set_ylabel('position (m)'); ax[0].legend(['x','y','z'],ncol=3); ax[1].plot(dist); ax[1].axhline(0,color='k'); ax[1].set_ylabel('signed plane distance (m)'); ax[1].set_xlabel('frame index'); fig.suptitle(f"Track 2 episode 0: {r['object']} | tier 1"); fig.tight_layout(); fig.savefig(paths['figs']/'track2_overview.png',dpi=140); plt.close(fig); print('sequence',r['sequence_id'],'rows',len(d),'mesh',m['mesh_exists'],'distance range',f"{dist.min():.4f} to {dist.max():.4f} m"); display(Image(filename=str(paths['figs']/'track2_overview.png')))

### Track 3 ego video and mocap visibility

**What it does.** Chooses the shortest public Track 3 episode, decodes ego camera A, filters visibility flags, and visualizes visibility over time.

**Input.** Track 3 Parquet, ego video, camera calibration, mesh/URDF inventory.

**Output.** `outputs/track3_manifest.json` and `outputs/figs/track3_overview.png`.

**References.** Track 3 README: `video_to_data_challenge/track_3/README.md`

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

t3=track3_paths(); r=min(t3,key=lambda x:len(read_parquet(x['parquet']))); d=read_parquet(r['parquet']); objs=decode_track3_objects(d); frames,vi=sample_video(r['videos']['ego_cam_a'],max_frames=12,resize_width=640); m={**r,'video_info':vi,'objects':{n:{'visible_frames':int(v['visible'].sum()),'total_frames':len(v['visible']),'visible_fraction':float(v['visible'].mean())} for n,v in objs.items()},'pose_convention':'world_T_object, [x,y,z,qw,qx,qy,qz], x/y episode-centred, z metric; visible=false pose is zero and filtered before use.'}; write_json(paths['root']/'track3_manifest.json',m); fig,ax=plt.subplots(figsize=(8,3.6)); [ax.plot(v['visible'].astype(int),label=n) for n,v in objs.items()]; ax.set_xlabel('frame index'); ax.set_ylabel('visible (1/0)'); ax.set_yticks([0,1]); ax.legend(ncol=3,fontsize=8); ax.set_title(f"Track 3 episode {r['episode_index']} visibility"); fig.tight_layout(); fig.savefig(paths['figs']/'track3_visibility.png',dpi=140); plt.close(fig); save_contact_sheet(frames,paths['figs']/'track3_overview.png',[f"frame {i}" for i in vi['sample_indices']]); print('episode',r['episode_index'],'task',r.get('task'),'frames',len(d),'objects',list(objs),'fps',f"{vi['fps']:.2f} Hz"); display(pd.DataFrame([{'object':n,'visible_frames':v['visible'].sum(),'total_frames':len(v['visible']),'visible_fraction':f"{v['visible'].mean()*100:.1f}%"} for n,v in objs.items()])); display(Image(filename=str(paths['figs']/'track3_overview.png')))

### Cross-track conventions and limitations

**What it does.** Prints the released coordinate and ground-truth coverage conventions.

**Input.** The three track READMEs and manifests.

**Output.** Printed table and audit JSON.

**References.** Local track READMEs; challenge site: https://nvidia-isaac.github.io/video_to_data/v2d_challenge/

In [ ]:
from pathlib import Path
import json, sys, time, platform, subprocess, importlib.util
import matplotlib; matplotlib.use('Agg', force=True)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'v2d_cpu_utils.py').exists(): ROOT=Path('/Users/luran/Desktop/V2D_Challenge')
sys.path.insert(0,str(ROOT))
from v2d_cpu_utils import *
paths=ensure_outputs()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'font.size':9})

conventions=pd.DataFrame([{'track':'Track 1','view':'third-person monocular RGB','fps':'30 Hz','pose/GT':'frame index only; no pose/mesh GT in local release','world frame':'not released'},{'track':'Track 2','view':'third-person RGB plus 4D trajectories','fps':'30 Hz','pose/GT':'SOMA-X human/object trajectories; tier 1 clean, tier 2 synthetic noise','world frame':'source trajectory / g1_retarget_world'},{'track':'Track 3','view':'ego_cam_a/b/c RGB','fps':'20.25 Hz nominal','pose/GT':'mocap object pose with visible flag; zero pose when invisible','world frame':'episode-centred x/y, metric z in mesh frame'}]); display(conventions); print('No Track 1 episode ID or object name is shared with Track 3 public episodes; later comparison reports representative per-track sequences and labels this limitation.'); print('Audit artifacts:',sorted(str(p.relative_to(paths['root'])) for p in paths['root'].glob('*') if p.is_file()))